# Gradus — Simulaciones del algoritmo Dual-Loop

Simulacion de 60 dias para 4 perfiles de estudiante realistas.
Todos los parametros vienen de `SM2Config`; no hay valores hardcodeados.

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import date, timedelta
from dataclasses import dataclass

from algorithm import (
    SM2Config, SM2ItemState, update_item_state,
    is_graduated, build_session,
    should_reinsert, FunctionFamily, SkillType, ItemKey, SessionItem,
)

print('Imports OK')

## Cinturones y catalogo

In [ ]:
BELT_ORDER = ['white', 'blue', 'violet', 'brown', 'black']

BELT_FUNCTIONS = {
    'white':  [FunctionFamily.LINEAR],
    'blue':   [FunctionFamily.QUADRATIC, FunctionFamily.POLYNOMIAL],
    'violet': [FunctionFamily.EXPONENTIAL, FunctionFamily.LOGARITHMIC],
    'brown':  [FunctionFamily.TRIGONOMETRIC],
    'black':  [FunctionFamily.RATIONAL],
}

BELT_COLORS = {
    'white':  '#d1d5db',
    'blue':   '#3b82f6',
    'violet': '#8b5cf6',
    'brown':  '#a16207',
    'black':  '#374151',
}

BELT_LABELS = {
    'white':  'Blanco',
    'blue':   'Azul',
    'violet': 'Violeta',
    'brown':  'Marron',
    'black':  'Negro',
}

FUNCTION_TO_BELT = {
    f: belt
    for belt, families in BELT_FUNCTIONS.items()
    for f in families
}

ORDERED_CATALOG = [
    ItemKey(function_family=family, skill_type=skill)
    for belt in BELT_ORDER
    for family in BELT_FUNCTIONS[belt]
    for skill in SkillType
]

BELT_THRESHOLDS = {}
cumulative = 0
for belt in BELT_ORDER:
    cumulative += len(BELT_FUNCTIONS[belt]) * len(list(SkillType))
    BELT_THRESHOLDS[belt] = cumulative

print(f'Catalogo: {len(ORDERED_CATALOG)} items')
for belt in BELT_ORDER:
    n = len(BELT_FUNCTIONS[belt]) * 3
    print(f'  {BELT_LABELS[belt]}: {n} items ({BELT_THRESHOLDS[belt]} acumulados)')

## Perfiles de estudiante

In [ ]:
@dataclass
class StudentProfile:
    name: str
    color: str
    sessions_per_week: int
    quality_probs: dict

    def sample_quality(self, rng):
        qualities = list(self.quality_probs.keys())
        probs = list(self.quality_probs.values())
        return int(rng.choice(qualities, p=probs))

    def practice_days(self, total_days):
        days = set()
        if self.sessions_per_week == 4:
            offsets = [0, 2, 4, 6]
        elif self.sessions_per_week == 3:
            offsets = [0, 2, 4]
        else:
            offsets = [0, 3]
        week = 0
        while True:
            for off in offsets:
                d = week * 7 + off + 1
                if d > total_days:
                    return days
                days.add(d)
            week += 1


PROFILES = [
    StudentProfile(
        name='Casi perfecto',
        color='#16a34a',
        sessions_per_week=4,
        quality_probs={1: 0.02, 3: 0.08, 4: 0.20, 5: 0.70},
    ),
    StudentProfile(
        name='Bueno',
        color='#2563eb',
        sessions_per_week=4,
        quality_probs={1: 0.10, 3: 0.15, 4: 0.35, 5: 0.40},
    ),
    StudentProfile(
        name='Promedio',
        color='#d97706',
        sessions_per_week=3,
        quality_probs={1: 0.30, 3: 0.30, 4: 0.25, 5: 0.15},
    ),
    StudentProfile(
        name='Por debajo',
        color='#dc2626',
        sessions_per_week=2,
        quality_probs={1: 0.60, 3: 0.20, 4: 0.12, 5: 0.08},
    ),
]

for p in PROFILES:
    p_fail = p.quality_probs[1]
    p_pass_session = 1 - p_fail ** 3
    print(f'{p.name}: {p.sessions_per_week} ses/sem | fallo/intento: {p_fail:.0%} | prob paso sesion: {p_pass_session:.1%}')

## Motor de simulacion

In [ ]:
def make_introduce_callback(items, requested):
    def introduce_new_item():
        for key in ORDERED_CATALOG:
            if key in items or key in requested:
                continue
            belt = FUNCTION_TO_BELT[key.function_family]
            belt_idx = BELT_ORDER.index(belt)
            if belt_idx > 0:
                prev_belt = BELT_ORDER[belt_idx - 1]
                prev_keys = [k for k in ORDERED_CATALOG if FUNCTION_TO_BELT[k.function_family] == prev_belt]
                if not all(k in items and items[k].phase == 'review' for k in prev_keys):
                    return None
            requested.add(key)
            return key
        return None
    return introduce_new_item


def simulate(profile, days=60, seed=42):
    rng = np.random.default_rng(seed)
    config = SM2Config()
    start_date = date(2026, 1, 1)

    items = {}
    practice_days = profile.practice_days(days)
    daily_graduated = []
    session_belt_counts = []
    graduation_days = {}
    n_graduated = 0

    for day in range(1, days + 1):
        today = start_date + timedelta(days=day - 1)

        if day not in practice_days:
            daily_graduated.append(n_graduated)
            continue

        requested_this_session = set()
        introduce_cb = make_introduce_callback(items, requested_this_session)
        session = build_session(items, today=today, config=config, introduce_new_item=introduce_cb)

        for si in session:
            if si.key not in items:
                items[si.key] = SM2ItemState(
                    phase=si.state.phase,
                    step_index=si.state.step_index,
                    ease_factor=si.state.ease_factor,
                    interval=si.state.interval,
                    repetitions=si.state.repetitions,
                    next_review=today,
                )

        queue = list(session)
        intra_counts = {}
        belt_counts = {b: 0 for b in BELT_ORDER}

        while queue:
            si = queue.pop(0)
            key = si.key
            current_state = items[key]
            quality = profile.sample_quality(rng)
            new_state = update_item_state(current_state, quality, config=config, today=today)
            items[key] = new_state

            belt = FUNCTION_TO_BELT[key.function_family]
            belt_counts[belt] += 1

            if new_state.phase == 'review' and current_state.phase == 'learning' and key not in graduation_days:
                graduation_days[key] = day
                n_graduated += 1

            if quality < config.quality_threshold_pass:
                count = intra_counts.get(key, 0)
                if should_reinsert(new_state, count, config=config):
                    intra_counts[key] = count + 1
                    queue.append(SessionItem(key=key, state=new_state))

        daily_graduated.append(n_graduated)
        session_belt_counts.append((day, belt_counts))

    white_keys = [k for k in ORDERED_CATALOG if FUNCTION_TO_BELT[k.function_family] == 'white']
    white_days = [graduation_days.get(k) for k in white_keys]
    time_to_blue = max(white_days) if all(d is not None for d in white_days) else None

    return {
        'daily_graduated': daily_graduated,
        'session_belt_counts': session_belt_counts,
        'graduation_days': graduation_days,
        'time_to_blue': time_to_blue,
        'items': items,
        'practice_days': practice_days,
    }

In [ ]:
DAYS = 60
results = {}
for profile in PROFILES:
    results[profile.name] = simulate(profile, days=DAYS, seed=42)

print('Resumen a 60 dias\n' + '-' * 50)
for profile in PROFILES:
    r = results[profile.name]
    tb = r['time_to_blue']
    grad = r['daily_graduated'][-1]
    n_sessions = len(r['session_belt_counts'])
    print(f"{profile.name}")
    print(f"  Sesiones  : {n_sessions}")
    print(f"  Graduados : {grad}/21")
    print(f"  Azul      : {'dia ' + str(tb) if tb else 'no alcanzado'}")
    print()

## Grafico 1 — Items graduados acumulados (60 dias)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Items graduados acumulados — 60 dias', fontsize=14, fontweight='bold')

days_x = list(range(1, DAYS + 1))

belt_ref = [
    (BELT_THRESHOLDS['white'],  '#9ca3af', 'Blanco (3)'),
    (BELT_THRESHOLDS['blue'],   '#3b82f6', 'Azul (9)'),
    (BELT_THRESHOLDS['violet'], '#8b5cf6', 'Violeta (15)'),
]

for ax, profile in zip(axes.flat, PROFILES):
    r = results[profile.name]
    grad = r['daily_graduated']

    ax.plot(days_x, grad, color=profile.color, linewidth=2.5, zorder=3)
    ax.axhline(y=21, color='#6b7280', linestyle='--', linewidth=0.9, alpha=0.7)
    ax.text(DAYS * 0.98, 21.4, 'Total (21)', ha='right', fontsize=7, color='#6b7280')

    for y, color, label in belt_ref:
        ax.axhline(y=y, color=color, linestyle=':', linewidth=0.8, alpha=0.6)
        ax.text(DAYS * 0.98, y + 0.4, label, ha='right', fontsize=7, color=color, alpha=0.9)

    for d in r['practice_days']:
        if d <= DAYS:
            ax.axvline(x=d, color=profile.color, linewidth=0.3, alpha=0.2, zorder=1)

    ax.set_xlim(1, DAYS)
    ax.set_ylim(0, 23)
    ax.set_title(profile.name, fontweight='bold', color=profile.color, fontsize=11)
    ax.set_xlabel('Dia')
    ax.set_ylabel('Items graduados')
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## Grafico 2 — Composicion de sesiones por cinturon (primeras 2 semanas)

In [ ]:
FIRST_N_DAYS = 14

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Ejercicios por sesion segun cinturon — primeras 2 semanas', fontsize=14, fontweight='bold')

legend_patches = [
    mpatches.Patch(color=BELT_COLORS[b], label=BELT_LABELS[b], edgecolor='#e5e7eb', linewidth=0.5)
    for b in BELT_ORDER
]

for ax, profile in zip(axes.flat, PROFILES):
    r = results[profile.name]

    day_belt = {d: {b: 0 for b in BELT_ORDER} for d in range(1, FIRST_N_DAYS + 1)}
    for day, counts in r['session_belt_counts']:
        if day <= FIRST_N_DAYS:
            day_belt[day] = counts

    days_x = list(range(1, FIRST_N_DAYS + 1))
    bottoms = [0] * FIRST_N_DAYS

    for belt in BELT_ORDER:
        values = [day_belt[d][belt] for d in days_x]
        ax.bar(
            days_x, values, bottom=bottoms,
            color=BELT_COLORS[belt],
            edgecolor='white', linewidth=0.5,
            width=0.7,
        )
        bottoms = [b + v for b, v in zip(bottoms, values)]

    for d in range(1, FIRST_N_DAYS + 1):
        if d not in r['practice_days']:
            ax.axvspan(d - 0.4, d + 0.4, alpha=0.04, color='#6b7280', zorder=0)

    ax.set_title(profile.name, fontweight='bold', color=profile.color, fontsize=11)
    ax.set_xlabel('Dia calendario')
    ax.set_ylabel('Ejercicios')
    ax.set_xticks(days_x)
    ax.set_xlim(0.4, FIRST_N_DAYS + 0.6)
    ax.set_ylim(0, 20)
    ax.grid(True, alpha=0.25, axis='y')

fig.legend(
    handles=legend_patches,
    loc='lower center',
    ncol=5,
    bbox_to_anchor=(0.5, -0.03),
    frameon=True,
    fontsize=10,
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## Grafico 3 — Tiempo hasta cinturon azul

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
fig.suptitle('Dias hasta obtener el cinturon azul', fontsize=14, fontweight='bold')

y_pos = list(range(len(PROFILES)))
bar_values = []
bar_colors = []
bar_labels = []

for profile in PROFILES:
    tb = results[profile.name]['time_to_blue']
    bar_values.append(tb if tb is not None else DAYS)
    bar_colors.append(profile.color)
    bar_labels.append(f'Dia {tb}' if tb is not None else f'No alcanzado ({DAYS} dias)')

bars = ax.barh(y_pos, bar_values, color=bar_colors, height=0.5, edgecolor='white', linewidth=0.8)

for bar, label, profile in zip(bars, bar_labels, PROFILES):
    ax.text(
        bar.get_width() + 0.8,
        bar.get_y() + bar.get_height() / 2,
        label,
        va='center', fontsize=11, fontweight='bold',
        color=profile.color,
    )

ax.set_yticks(y_pos)
ax.set_yticklabels([p.name for p in PROFILES], fontsize=11)
ax.set_xlabel('Dias desde el inicio')
ax.set_xlim(0, DAYS + 16)
ax.invert_yaxis()

for week, style in [(7, '--'), (14, ':'), (21, ':')]:
    ax.axvline(x=week, color='#9ca3af', linestyle=style, linewidth=0.9, alpha=0.7)
    ax.text(week, len(PROFILES) - 0.1, f'Sem {week // 7}', ha='center', fontsize=8, color='#9ca3af')

ax.grid(True, alpha=0.25, axis='x')
plt.tight_layout()
plt.show()